# TARDIS - Modèle de Régression Linéaire

Ce notebook implémente un modèle de régression linéaire simple pour prédire les retards de trains. Conformément au sujet, nous comparons les performances à une baseline (prédiction par la moyenne).

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Chargement des données
df = pd.read_csv("cleaned_dataset.csv")
df['Date'] = pd.to_datetime(df['Date'])
print("Données chargées avec succès.")

Données chargées avec succès.


## 1. Préparation des variables (Feature Engineering)

Nous sélectionnons des variables numériques et catégorielles pour alimenter notre régression linéaire.

In [2]:
df['day_of_week'] = df['Date'].dt.dayofweek

# Cible (Target)
target = "Retard moyen de tous les trains à l'arrivée"

# Caractéristiques (Features)
features_cat = ["Gare de départ", "Gare d'arrivée", "Service"]
features_num = ["Année", "Mois", "day_of_week", "Durée moyenne du trajet", "Nombre de circulations prévues"]

X = df[features_cat + features_num]
y = df[target]

# Division Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 2. Construction du Pipeline de Régression Linéaire

La régression linéaire nécessite que les variables catégorielles soient encodées (One-Hot Encoding) et que les variables numériques soient idéalement normalisées.

In [3]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), features_num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), features_cat),
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)
print("Modèle entraîné.")

Modèle entraîné.


## 3. Évaluation du Modèle vs Baseline

Nous calculons la MAE, la RMSE et le R²[cite: 73]. Nous vérifions si le modèle bat la baseline (prédiction par la moyenne des retards).

In [4]:
y_pred = model.predict(X_test)

# Métriques du modèle
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# Baseline (Moyenne simple)
baseline_pred = np.full_like(y_test, y_train.mean())
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print("--- Résultats de l'évaluation ---")
print(f"MAE Modèle : {mae:.2f} min")
print(f"MAE Baseline : {baseline_mae:.2f} min")
print(f"RMSE : {rmse:.2f} min")
print(f"R² Score : {r2:.2f}")

if mae < baseline_mae:
    print("\n✅ Le modèle est plus performant que la baseline.")
else:
    print("\n⚠️ Le modèle n'est pas plus performant que la baseline.")

--- Résultats de l'évaluation ---
MAE Modèle : 2.47 min
MAE Baseline : 3.08 min
RMSE : 3.88 min
R² Score : 0.24

✅ Le modèle est plus performant que la baseline.


## 4. Sauvegarde du Modèle

Exportation du fichier `model.joblib` pour intégration dans le dashboard Streamlit[cite: 79, 108].

In [5]:
joblib.dump(model, "model.joblib")
print("Modèle sauvegardé sous 'model.joblib'.")

Modèle sauvegardé sous 'model.joblib'.


Le choix d'un modèle d'arbre (comme le Random Forest) s'impose car les retards de la SNCF ne suivent pas une logique linéaire et proportionnelle.  Alors qu'une régression linéaire s'obstine à tracer une ligne droite là où le chaos règne, les arbres découpent intelligemment les données historiques pour capturer des interactions complexes entre les gares et les périodes de forte affluence. En créant des embranchements logiques, ce modèle comprend que dix minutes de retard un lundi matin n'ont pas la même dynamique qu'un dimanche soir, vous permettant de livrer des prédictions bien plus fiables à vos milliers de futurs utilisateurs.